# MedGemma-4B QLoRA 파인튜닝

## Google Colab 무료 T4 환경용

`google/medgemma-4b-it`를 QLoRA 방식으로 파인튜닝하는 Notebook입니다.

### 주요 구성

* MedGemma 4B Instruct
* 4bit 양자화
* LoRA / QLoRA
* Google Colab T4 GPU
* KorMedMCQA doctor 데이터셋
* Google Drive 체크포인트 저장
* Hugging Face Hub LoRA Adapter 업로드

### 실행 순서

반드시 아래 순서대로 실행하세요.

1. GPU 확인
2. 패키지 설치
3. 런타임 재시작
4. 환경 확인
5. Hugging Face 로그인
6. 데이터셋 로드
7. 데이터셋 포맷팅
8. MedGemma 로드
9. LoRA 설정
10. Google Drive 연결
11. 학습
12. 필요하면 체크포인트에서 재개
13. 추론 테스트
14. Hugging Face Hub 업로드

### 중요

패키지 설치 셀 실행 후에는 반드시:

`런타임 → 세션 다시 시작`

또는

`런타임 → 런타임 다시 시작`

을 실행하세요.

Colab에서 이미 로드된 NumPy와 새로 설치된 NumPy가 충돌하는 것을 방지하기 위한 과정입니다.


---

# 0. GPU 확인


In [ ]:
!nvidia-smi


정상적으로 T4 GPU가 연결되어 있다면 다음과 비슷한 정보가 출력됩니다.

```text
Tesla T4
```

GPU가 없다면 이후 MedGemma 학습을 진행하지 마세요.

---

# 1. 패키지 설치

## 중요 (numpy는 건드리지 않습니다)

한때 `numpy.dtype size changed, may indicate binary incompatibility` 에러 때문에
numpy를 특정 버전(1.26.4)으로 낮춰 고정했었는데, 이게 오히려 문제였습니다 — 지금 Colab
기본 이미지는 `opencv`, `jax`, `cupy`, `shap`, `cudf`, `tifffile`, `rasterio` 등
수십 개 사전 설치 패키지가 전부 `numpy>=2`를 요구하도록 이미 맞춰져 있어서, numpy를 2.0
밑으로 내리면 오히려 그 패키지들과 어긋나 같은 종류의 ABI 에러가 재발합니다.

그래서 **numpy 버전은 아예 지정하지 않고, Colab에 이미 깔린 걸 그대로 씁니다.** 우리가 필요한
패키지만 설치합니다.


In [ ]:
%pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub trl

# torchvision은 이 노트북(텍스트 전용 QLoRA 파인튜닝)에 필요 없음.
# Colab 기본 이미지의 torch/torchvision 버전이 서로 안 맞는 경우가 있어서
# (torch 2.13.0 vs torchvision이 요구하는 torch 2.11.0), 남겨두면 MedGemma처럼
# 멀티모달 모델 클래스를 로드할 때 torchvision::nms 관련 에러로 막힌다. 지워서 회피.
!pip uninstall -y -q torchvision


---

# 2. 반드시 런타임 재시작

## 이 셀은 설명용입니다.

패키지 설치가 완료되면 아래 메뉴를 직접 실행하세요.

```text
런타임
→ 런타임 다시 시작
```

또는 Colab UI에 따라:

```text
런타임
→ 세션 다시 시작
```

재시작 후 아래 셀부터 다시 실행합니다.

---

# 3. Python / NumPy / GPU 환경 확인


In [ ]:
import sys
import numpy
import torch

print("Python :", sys.version)
print("NumPy  :", numpy.__version__)
print("PyTorch:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


정상적인 예 (numpy 버전은 Colab이 준 값 그대로면 됨, 특정 값을 강제하지 않음):

```text
CUDA available: True
GPU: Tesla T4
```

---

# 4. 핵심 패키지 Import 테스트

앞서 겪었던 numpy ABI 충돌이 없는지 먼저 확인합니다. 여기서 에러가 나면 numpy를 건드리는
다른 셀/명령을 실행한 적이 있는지 먼저 의심하고, `런타임 다시 시작` 후 재시도하세요.


In [ ]:
import numpy
import pandas
import pyarrow
import scipy

print("NumPy   :", numpy.__version__)
print("Pandas  :", pandas.__version__)
print("PyArrow :", pyarrow.__version__)
print("SciPy   :", scipy.__version__)

from datasets import load_dataset

print("datasets import OK")


여기서:

```text
datasets import OK
```

가 출력되어야 합니다.

---

# 5. 패키지 충돌 검사


In [ ]:
!pip check


여기서 의존성 문제가 출력되면 내용을 확인하세요.

특히 다음과 같은 패키지에서 문제가 없어야 합니다.

```text
numpy
pandas
pyarrow
scipy
datasets
transformers
peft
trl
bitsandbytes
```

---

# 6. Hugging Face 로그인

MedGemma를 사용하기 전에 Hugging Face에서 MedGemma 접근 권한을 승인하고 Token을 준비해야 합니다.

Colab:

```text
왼쪽 메뉴
→ 열쇠 아이콘
→ Secrets
→ HF_TOKEN
```

으로 등록하세요.

코드에 Token을 직접 입력하지 않습니다.


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN이 없습니다. "
        "Colab 왼쪽 메뉴의 Secrets에 HF_TOKEN을 등록하세요."
    )

login(token=HF_TOKEN)

print("Hugging Face login OK")


---

# 8. 데이터셋 로드

사용 데이터셋:

```text
sean0042/KorMedMCQA
```

직종별 config:

```text
doctor
nurse
pharm
dentist
```

현재는 의사 시험 데이터셋을 사용합니다.


In [ ]:
from datasets import load_dataset

DATASET_ID = "sean0042/KorMedMCQA"
DATASET_CONFIG = "doctor"

dataset = load_dataset(
    DATASET_ID,
    name=DATASET_CONFIG,
    split="train"
)

print(dataset)
print(dataset.column_names)


---

# 9. 데이터셋 샘플 확인


In [ ]:
print(dataset[0])
print("Dataset size:", len(dataset))


---

# 10. 데이터셋 컬럼 확인


In [ ]:
print(dataset.column_names)


---

# 10-1. 데이터셋 정제 (결측/중복 제거)

이전 학습에서 나온 "같은 글자 반복" 증상의 원인을 데이터 쪽에서 줄여보기 위한 정제 단계입니다.

**먼저 확인해야 할 사실**: KorMedMCQA는 `cot`(해설) 컬럼이 있긴 하지만, 이건 `fewshot`
split(5개)에만 채워져 있고 우리가 학습에 쓰는 `train` split(1,890개)에는 사실상 비어
있습니다. 즉 "해설이 있는 데이터만 골라서 학습"은 이 데이터셋 구조상 불가능합니다 — 정답이
몇 단어로 끝나는 정형화된 패턴 자체는 필터링으로 바뀌지 않습니다.

여기서 실제로 하는 것:

* 질문/선택지가 비어 있거나 선택지가 1개 이하인 결측 행 제거
* 완전히 같은 문제(question)가 중복으로 들어있으면 첫 등장만 남기고 제거 — 같은 정형 패턴을
  반복 학습해서 과적합을 키우는 걸 줄이기 위함
* 공백/개행 정규화

근본적인 "정답이 짧고 정형화됨" 문제는 아래 11번(포맷팅)에서 정답을 완전한 문장으로 바꾸고,
24-1번에서 정답 부분에만 loss를 집중시키는 방식으로 보완합니다.


In [ ]:
import re

_OPTION_KEYS = ["A", "B", "C", "D", "E"]


def _normalize_ws(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def _is_valid_row(example):
    if not _normalize_ws(example.get("question")):
        return False

    non_empty_options = [
        example.get(key)
        for key in _OPTION_KEYS
        if example.get(key) is not None and _normalize_ws(example.get(key))
    ]
    if len(non_empty_options) < 2:
        return False

    if example.get("answer") in (None, ""):
        return False

    return True


before_n = len(dataset)

dataset = dataset.filter(_is_valid_row)
after_filter_n = len(dataset)

# 완전히 같은 문제(question)가 중복으로 들어있으면 첫 등장만 남긴다.
seen_questions = set()
keep_indices = []
for i, q in enumerate(dataset["question"]):
    key = _normalize_ws(q)
    if key in seen_questions:
        continue
    seen_questions.add(key)
    keep_indices.append(i)

dataset = dataset.select(keep_indices)
after_dedup_n = len(dataset)

if "cot" in dataset.column_names:
    cot_present = sum(1 for c in dataset["cot"] if c is not None and _normalize_ws(c))
else:
    cot_present = 0

print(f"원본: {before_n}개")
print(f"결측/형식 오류 제거 후: {after_filter_n}개 ({before_n - after_filter_n}개 제거)")
print(f"중복 문항 제거 후: {after_dedup_n}개 ({after_filter_n - after_dedup_n}개 제거)")
print(f"해설(cot)이 채워진 항목: {cot_present}개 / {after_dedup_n}개")


예상되는 주요 컬럼:

```text
question
A
B
C
D
E
answer
```

`cot`이 존재하는 경우에는 해설도 사용할 수 있습니다.

---

# 11. 데이터셋 포맷팅

KorMedMCQA 데이터를 MedGemma의 대화 형식으로 변환합니다.


In [ ]:
OPTION_KEYS = ["A", "B", "C", "D", "E"]

# 여러 출처(KorMedMCQA + 대화형 데이터셋)를 섞어 학습시킬 때 톤/제약을 통일하기 위한 공통
# 지시문. "두개내과", "소화기내시경센터"처럼 실존하지 않는 진료과명을 답변에 넣는 문제를
# 줄이려고, 실제 존재하는 전문과목 이름만 쓰라고 명시해둔다.
INSTRUCTION_PREFIX = (
    "당신은 신중하고 정확한 의료 지식을 갖춘 병의원 진료상담 챗봇입니다. 확정적인 진단이나 "
    "처방 대신 가능성과 권장 사항을 안내하고, 진료과 이름은 내과·외과·소아청소년과·산부인과·"
    "신경과·신경외과·정신건강의학과·정형외과·이비인후과·피부과·안과·비뇨의학과·응급의학과·"
    "가정의학과 등 실제 존재하는 전문과목 명칭만 사용하세요."
)


def format_example(example):
    options_text = "\n".join(
        f"{key}. {_normalize_ws(example[key])}"
        for key in OPTION_KEYS
        if example.get(key) is not None
        and _normalize_ws(example[key])
    )

    user_turn = (
        f"{INSTRUCTION_PREFIX}\n\n"
        f"{_normalize_ws(example['question'])}\n\n"
        f"선택지:\n"
        f"{options_text}"
    )

    answer_value = example["answer"]

    # answer가 문자열로 들어오는 경우도 대응
    if isinstance(answer_value, str):
        answer_value = answer_value.strip()

        if answer_value.upper() in OPTION_KEYS:
            answer_letter = answer_value.upper()
        else:
            answer_letter = OPTION_KEYS[int(answer_value) - 1]
    else:
        answer_letter = OPTION_KEYS[int(answer_value) - 1]

    answer_text = _normalize_ws(example.get(answer_letter, ""))

    # "정답: C. 내용" 같은 전보체 대신 완전한 문장으로 — KorMedMCQA train split엔 cot(해설)이
    # 없어서(10-1번 참고) 이게 사실상 유일하게 "자연스러운 문장 흐름"을 학습시킬 수 있는
    # 지점이다. 타깃이 몇 단어짜리 정형 패턴에만 머물수록, 학습 후 자유 서술형 질문에는 그
    # 좁은 패턴을 벗어나지 못하고 반복 생성에 빠지기 쉽다.
    model_turn = f"정답은 {answer_letter}번, {answer_text}입니다."

    cot = example.get("cot")

    if cot is not None and _normalize_ws(cot):
        model_turn += f"\n\n해설: {_normalize_ws(cot)}"

    # trl의 최신 SFTTrainer는 DataCollatorForCompletionOnlyLM이 없어지고(24-1번 참고),
    # 대신 prompt/completion을 분리한 "conversational prompt-completion" 포맷을 표준으로
    # 요구한다 — SFTConfig(completion_only_loss=True)가 이 포맷에서만 동작한다. trl은
    # role로 "system"/"user"/"assistant"/"tool"만 받는다("model"은 ValueError) — Gemma
    # 자체 chat_template이 내부적으로 "assistant"를 "<start_of_turn>model"로 바꿔주므로
    # 여기서는 표준 role명 그대로 "assistant"를 쓴다.
    return {
        "prompt": [{"role": "user", "content": user_turn}],
        "completion": [{"role": "assistant", "content": model_turn}],
    }


---

# 12. 데이터셋 변환


In [ ]:
dataset = dataset.map(
    format_example,
    remove_columns=[
        column
        for column in dataset.column_names
        if column != "text"
    ]
)


---

# 13. 변환 결과 확인


In [ ]:
print(dataset[0])


---

# 13-1. 대화형 데이터셋 혼합 (GenMedGPT-5k-ko)

KorMedMCQA만으로 학습하면 사용자 턴이 **항상** "질문 + 선택지" 형태다. 그런데 실제 상담은
선택지 없이 증상만 던지는 경우가 대부분이라, 그런 입력은 학습 때 한 번도 못 본 모양이 되어
답변이 한 단어로 붕괴하거나("소화기내시경센터") 반대로 통제 안 된 장문으로 새는 등 불안정해진다.

`ChuGyouk/GenMedGPT-5k-ko`(MIT 라이선스, ChatDoctor 데이터를 한국어로 번역)는 정확히
"환자 증상 설명(선택지 없음) → 의사 답변" 형태라 이 빈틈을 메운다. KorMedMCQA와 규모가
비슷하도록(10-1번 정제 후 ~1,800개) 샘플링해서, 한쪽이 학습을 지배하지 않게 한다. 두 출처
모두 위에서 정의한 `INSTRUCTION_PREFIX`를 동일하게 붙여서 톤과 "실존 진료과명만 사용" 제약을
통일한다.


In [ ]:
from datasets import concatenate_datasets

GENMEDGPT_ID = "ChuGyouk/GenMedGPT-5k-ko"
genmedgpt = load_dataset(GENMEDGPT_ID, split="train")
print(genmedgpt)
print(genmedgpt[0])


def format_genmedgpt(example):
    symptom = _normalize_ws(example.get("input"))
    response = _normalize_ws(example.get("output"))
    if not symptom or not response:
        return {"prompt": [], "completion": []}

    user_turn = f"{INSTRUCTION_PREFIX}\n\n{symptom}"

    return {
        "prompt": [{"role": "user", "content": user_turn}],
        "completion": [{"role": "assistant", "content": response}],
    }


# 전량(5,451개)을 다 섞으면 KorMedMCQA(정제 후 ~1,800개)를 압도해서 학습 시간도 크게
# 늘어난다 — 규모를 비슷하게 맞춰서 두 스타일이 균형 있게 섞이도록 샘플링한다. 더 많이
# 섞고 싶으면 이 숫자를 늘리면 되는데, Colab 무료 T4 세션 시간 제한을 염두에 둬야 한다.
GENMEDGPT_SAMPLE_N = min(2000, len(genmedgpt))
genmedgpt = genmedgpt.shuffle(seed=42).select(range(GENMEDGPT_SAMPLE_N))

genmedgpt = genmedgpt.map(
    format_genmedgpt,
    remove_columns=[c for c in genmedgpt.column_names if c not in ("prompt", "completion")],
)
genmedgpt = genmedgpt.filter(lambda ex: bool(ex["prompt"]) and bool(ex["completion"]))

print(f"\nGenMedGPT-5k-ko 변환 후: {len(genmedgpt)}개")
print(genmedgpt[0])

print(f"\nKorMedMCQA(선택지 있음): {len(dataset)}개")
print(f"GenMedGPT-5k-ko(자유 서술형, 선택지 없음): {len(genmedgpt)}개")

dataset = concatenate_datasets([dataset, genmedgpt]).shuffle(seed=42)
print(f"혼합 후 전체 학습 데이터: {len(dataset)}개")


---

# 13-2. 합친 데이터셋 저장/공유 (팀원 비교용)

지금까지 만든 `dataset`(KorMedMCQA + GenMedGPT-5k-ko 혼합, 셔플까지 끝난 상태)은 이 Colab
세션 메모리에만 있고 어디에도 저장되지 않는다 — 세션이 끊기면 사라지고, 다시 만들어도 셔플
시드나 업스트림 데이터셋 변경 때문에 완전히 같다는 보장이 없다.

**여러 모델(medgemma/gemma/qwen/llama)을 공정하게 비교하려면 전부 같은 데이터 스냅샷으로
학습해야 한다** — 그래서 지금 이 시점의 `dataset`을 얼려서 HF Hub에 비공개로 올려둔다.
팀원이 합친 데이터셋을 비교하고 싶다고 했으면, 이 리포지토리 이름을 공유하면 된다.


In [ ]:
MIXED_DATASET_REPO = "gon-0130/medgemma-mixed-dataset-v1"

dataset.push_to_hub(MIXED_DATASET_REPO, private=True)

print(f"혼합 데이터셋 업로드 완료: https://huggingface.co/datasets/{MIXED_DATASET_REPO}")
print(f"불러올 때: load_dataset('{MIXED_DATASET_REPO}')")
print(
    "팀원과 비교할 땐 이 리포지토리 이름과 링크를 공유하면 됩니다 — private 리포라서 "
    "HF 계정에 접근 권한(Settings > Collaborators)을 따로 추가해줘야 팀원이 열어볼 수 있습니다."
)


예상 형태:

```python
{
    "prompt": [{"role": "user", "content": "문제 내용\n\n선택지:\nA. ...\n..."}],
    "completion": [{"role": "assistant", "content": "정답은 C번, ...입니다."}],
}
```

`<start_of_turn>...` 같은 특수토큰을 직접 안 써도, SFTTrainer가 이 conversational
prompt-completion 포맷을 보면 자동으로 `tokenizer.apply_chat_template()`을 적용해서
Gemma 고유 포맷으로 감싸준다. role은 trl이 받는 표준 이름인 `"user"`/`"assistant"`를 쓰고,
`"assistant"`는 Gemma의 chat_template 내부에서 자동으로 `<start_of_turn>model`로
바뀐다("model"을 직접 role로 넣으면 trl이 `ValueError`를 낸다).

---

# 14. MedGemma 모델 설정


In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_ID = "google/medgemma-4b-it"


---

# 15. 4bit 양자화 설정

## T4에서는 float16을 사용합니다.

T4(Turing 아키텍처)는 bf16 텐서코어 가속을 지원하지 않습니다(Ampere 이상부터 지원). 그래서
T4 환경에서 안정적/효율적으로 쓰려면 BF16 대신 FP16을 씁니다.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


---

# 16. Tokenizer 로드


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


---

# 17. MedGemma 모델 로드


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,  # transformers 최신 버전은 torch_dtype 대신 dtype을 씀
    token=HF_TOKEN,
)


---

# 18. 모델 메모리 상태 확인


In [ ]:
print("Model loaded successfully")

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

    print(
        "GPU memory reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )


---

# 19. LoRA 설정


In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


---

# 20. K-bit Training 준비


In [ ]:
model = prepare_model_for_kbit_training(model)

model.config.use_cache = False


---

# 21. LoRA Configuration


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)


---

# 22. LoRA 적용


In [ ]:
model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()


출력 결과에서 전체 파라미터 대비 trainable parameter가 매우 적게 나오는 것이 정상입니다.

---

# 23. Google Drive 연결

Colab 세션이 종료되면 `/content`에 저장된 파일이 사라질 수 있습니다.

따라서 학습 체크포인트를 Google Drive에 저장합니다.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CKPT_DIR = (
    "/content/drive/MyDrive/"
    "medgemma-lora-ckpt"
)

print("Checkpoint directory:", CKPT_DIR)


---

# 24. 학습 설정

## T4 16GB 기준


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=CKPT_DIR,

    # T4 16GB
    per_device_train_batch_size=1,

    # 실제 batch size를 늘리는 효과
    gradient_accumulation_steps=8,

    # 1 epoch로는 정답이 짧고 정형화된 패턴(예: "정답은 C번, ...입니다")에 노출되는
    # 총 스텝 수가 적어서, 자유 서술형 질문에 그 패턴을 못 벗어나고 반복 생성에 빠지기 쉬웠다.
    # 정제된 데이터(10-1번)는 약 1,800개 정도라 2 epoch도 T4 무료 세션에서 충분히 감당된다.
    # 그래도 반복 증상이 남으면 3까지 올려보되, 과적합 신호(loss가 비정상적으로 빨리 0에
    # 가까워짐)가 보이면 다시 줄인다.
    num_train_epochs=2,

    learning_rate=2e-4,

    # T4는 bf16 텐서코어가 없어서 bf16=False. fp16=True(AMP GradScaler)는 Gemma 계열에서
    # LoRA 레이어 일부가 bfloat16으로 생성되는 경우가 있어 GradScaler가 그 텐서를 처리 못 해
    # "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16" 에러가 남.
    # 4bit 베이스 자체가 이미 압축돼있고 LoRA 파라미터는 작아서, AMP 없이 기본 정밀도로 학습.
    fp16=False,
    bf16=False,

    # 메모리 절약
    gradient_checkpointing=True,

    logging_steps=10,

    # Colab 세션 종료 대비
    save_steps=50,
    save_total_limit=3,

    # dataset이 이제 text 단일 필드가 아니라 prompt/completion으로 나뉜 conversational
    # 포맷이라(24-1번 참고) dataset_text_field는 더 안 씀. completion_only_loss는
    # prompt-completion 포맷에서 기본값이 True라 안 적어도 되지만, 명시적으로 남겨둔다.
    completion_only_loss=True,
    max_length=1024,

    # W&B 등의 외부 로깅 방지
    report_to="none",

    # 데이터 packing
    packing=False,
)


---

# 24-1. 정답 부분에만 Loss 집중시키기 (Completion-only Loss)

지금까지는 `text` 필드 전체(질문+선택지+정답)에 대해 언어모델 loss를 계산했다. 그런데 실제로
잘 배우길 원하는 건 "정답을 어떻게 생성하는가"이지, 매번 주어지는 질문/선택지 텍스트를 다시
예측하는 게 아니다. 질문 부분까지 loss에 포함되면 학습 신호가 흐려지고, 정답이 몇 단어로
끝나는 이 데이터에서는 정답 부분의 상대적 비중이 더 작아져서 반복 생성 경향을 키우는 요인이
될 수 있다.

**(업데이트) `DataCollatorForCompletionOnlyLM`은 최신 trl에서 제거됐다** — 대신
`SFTConfig(completion_only_loss=True)`를 쓰면 되는데, 이건 `text` 단일 필드가 아니라
**prompt/completion으로 나뉜 데이터셋**에서만 동작한다. 그래서 11번(포맷팅)과 13-1번의
`format_example`/`format_genmedgpt`가 `{"prompt": [...], "completion": [...]}` 형태로
반환하도록 이미 바꿔뒀다 — 이 셀은 별도 collator를 만드는 대신, 그 포맷이 제대로 됐는지만
확인한다.


In [ ]:
# prompt/completion 포맷이 제대로 됐는지, chat template이 실제로 적용되는지 확인한다.
sample = dataset[0]
assert "prompt" in dataset.column_names and "completion" in dataset.column_names, (
    "dataset에 prompt/completion 컬럼이 없다 — 11번/13-1번 포맷팅 함수가 최신 버전으로 "
    "적용됐는지 확인(런타임을 재시작하지 않고 예전 셀 결과가 남아있는 경우 이럴 수 있음)."
)

rendered = tokenizer.apply_chat_template(
    sample["prompt"] + sample["completion"],
    tokenize=False,
)
print("apply_chat_template 렌더링 결과:\n")
print(rendered)


---

# 25. SFTTrainer 생성


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)


---

# 26. 학습 전 Trainer 확인


In [ ]:
print(trainer)


---

# 27. 학습 시작

## 처음에는 1 epoch로 테스트하는 것을 권장합니다.


In [ ]:
trainer.train()


학습이 정상적으로 시작되면 다음과 비슷한 로그가 출력됩니다.

```text
***** Running training *****
Num examples = ...
Num Epochs = 1
...
```

---

# 28. 학습 결과 저장

학습이 정상적으로 끝난 후 LoRA Adapter를 저장합니다.


In [ ]:
FINAL_ADAPTER_DIR = (
    "/content/drive/MyDrive/"
    "medgemma-lora-final"
)

trainer.save_model(FINAL_ADAPTER_DIR)

tokenizer.save_pretrained(
    FINAL_ADAPTER_DIR
)

print(
    "Adapter saved to:",
    FINAL_ADAPTER_DIR
)


---

# 29. 학습 체크포인트 확인


In [ ]:
import os

print(os.listdir(CKPT_DIR))


---

# 30. 세션이 끊긴 경우 체크포인트에서 재개

Colab 세션이 종료된 경우:

1. 런타임 재연결
2. 패키지 설치
3. 런타임 재시작
4. HF 로그인
5. 데이터셋 로드
6. 모델 로드
7. LoRA 설정
8. Drive mount
9. Trainer 생성

까지 다시 실행합니다.

그 다음:


In [ ]:
import os

checkpoint_dirs = [
    os.path.join(CKPT_DIR, name)
    for name in os.listdir(CKPT_DIR)
    if name.startswith("checkpoint-")
]

checkpoint_dirs = [
    path
    for path in checkpoint_dirs
    if os.path.isdir(path)
]

if checkpoint_dirs:
    checkpoint_dirs.sort(
        key=lambda path: int(
            os.path.basename(path).split("-")[-1]
        )
    )

    latest_checkpoint = checkpoint_dirs[-1]

    print(
        "Latest checkpoint:",
        latest_checkpoint
    )

    trainer.train(
        resume_from_checkpoint=latest_checkpoint
    )

else:
    print("체크포인트가 없습니다. 처음부터 학습을 시작하세요.")


---

# 31. 간단한 추론 테스트

학습이 완료되었으면 먼저 Colab 안에서 모델이 제대로 답변하는지 테스트합니다.


추론 전에 학습 때 꺼뒀던 캐시를 다시 켭니다 — 안 켜면 두통이 3일째 있어요 같은
질문에 같은 글자만 반복하는("두두두두...") 증상이 날 수 있습니다.


In [ ]:
model.eval()
model.config.use_cache = True
model.gradient_checkpointing_disable()


In [ ]:
prompt = (
    "<start_of_turn>user\n"
    "두통이 3일째 있어요. "
    "어떤 진료과를 방문하는 것이 좋을까요?"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
    )

result = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
)

print(result)


---

# 32. 다른 질문으로 테스트


In [ ]:
test_questions = [
    "3일째 열이 나고 기침이 있어요.",
    "가슴이 갑자기 아프고 숨쉬기가 힘들어요.",
    "복통이 계속되는데 어느 진료과에 가야 하나요?",
]

for question in test_questions:

    prompt = (
        "<start_of_turn>user\n"
        f"{question}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    result = tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )

    print("=" * 80)
    print("질문:", question)
    print(result)


---

# 32-1. 정답 정확도 검증 (held-out test set)

지금까지 31~32번은 사람이 답을 눈으로 훑어보는 것뿐이었다 — 이건 "그럴듯해 보인다"는 인상이지
검증이 아니다. 실제로 "정답에 근접한지"를 수치로 확인하려면 **학습에 한 번도 안 쓴 정답 세트**가
필요하다.

KorMedMCQA의 `test` split(435개)이 정확히 이 용도다 — 우리는 `train`(1,890개)만 학습에
썼고 `test`는 지금까지 한 번도 안 건드렸으니 순수한 held-out 검증셋이다. 여기서 모델이 생성한
답변에 **정답 선택지 글자(A~E) 또는 정답 텍스트**가 포함되는지로 채점한다.

**주의**: 이 방식은 "사실을 정확히 골랐는가"만 측정한다 — 어투/공감 표현/면책 문구 같은 상담
스타일은 이 지표로 안 잡힌다(그건 사람이 직접 몇 개 읽어보거나, 더 강한 모델로 채점하는
방식이 필요 — 별개 문제다). 전체 435개를 다 돌리면 시간이 걸려서 우선 `EVAL_SAMPLE_N`개만
샘플링한다.


In [ ]:
import re as _re

eval_dataset = load_dataset(DATASET_ID, name=DATASET_CONFIG, split="test")
print(f"test split 크기: {len(eval_dataset)}개 (학습에 안 쓴 held-out)")

EVAL_SAMPLE_N = min(100, len(eval_dataset))  # 전부(435개) 돌리려면 len(eval_dataset)로 바꾸면 됨


def _eval_gold_answer(example):
    answer_value = example["answer"]
    if isinstance(answer_value, str):
        answer_value = answer_value.strip()
        letter = answer_value.upper() if answer_value.upper() in OPTION_KEYS else OPTION_KEYS[int(answer_value) - 1]
    else:
        letter = OPTION_KEYS[int(answer_value) - 1]
    return letter, _normalize_ws(example.get(letter, ""))


def _eval_prompt(example):
    options_text = "\n".join(
        f"{key}. {_normalize_ws(example[key])}"
        for key in OPTION_KEYS
        if example.get(key) is not None and _normalize_ws(example[key])
    )
    return (
        "<start_of_turn>user\n"
        f"{_normalize_ws(example['question'])}\n\n선택지:\n{options_text}"
        "<end_of_turn>\n<start_of_turn>model\n"
    )


correct = 0
wrong_examples = []

for example in eval_dataset.select(range(EVAL_SAMPLE_N)):
    prompt = _eval_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    gold_letter, gold_text = _eval_gold_answer(example)
    # 정답 글자를 답변 맨 앞부분에 언급했거나, 정답 텍스트 자체를 포함하면 정답으로 채점
    mentioned_letter = _re.search(rf"\b{gold_letter}\b", response[:20]) is not None
    mentioned_text = bool(gold_text) and gold_text in response
    is_correct = mentioned_letter or mentioned_text

    correct += int(is_correct)
    if not is_correct:
        wrong_examples.append({
            "question": example["question"][:60],
            "gold": f"{gold_letter}. {gold_text}",
            "response": response[:150],
        })

accuracy = correct / EVAL_SAMPLE_N
print(f"\n정확도: {correct}/{EVAL_SAMPLE_N} = {accuracy:.1%}")
print(f"\n틀린 예시 (최대 5개):")
for w in wrong_examples[:5]:
    print(f"- 질문: {w['question']}...")
    print(f"  정답: {w['gold']}")
    print(f"  모델 답변: {w['response']}...")
    print()


---

# 33. Hugging Face Hub에 LoRA Adapter 업로드

## 먼저 Hugging Face에서 업로드할 Repository를 준비하세요.

예:

```text
your-hf-account/medgemma-4b-lora-consultation
```

아래의 `<HF계정>`을 실제 계정명으로 변경합니다.


In [ ]:
ADAPTER_REPO = (
    "gon-0130/medgemma-4b-lora-consultation-v2"
)


---

# 34. Adapter 업로드


In [ ]:
model.push_to_hub(
    ADAPTER_REPO,
    private=True,
)

tokenizer.push_to_hub(
    ADAPTER_REPO,
    private=True,
)

print(
    "Uploaded to Hugging Face:",
    ADAPTER_REPO
)


---

# 35. 중요: GitHub와 Hugging Face의 역할

이 Notebook에서 GitHub는 학습 코드 관리용입니다.

```text
GitHub
├── train_medgemma_lora.ipynb
├── training code
└── configuration
```

학습된 모델 Adapter는 Hugging Face에 저장합니다.

```text
Hugging Face
└── medgemma-4b-lora-consultation
    ├── adapter_config.json
    ├── adapter_model.safetensors
    └── tokenizer files
```

GitHub에 대용량 모델 파일을 직접 올리지 않습니다.

---

# 36. 이후 FastAPI에서 사용하는 구조

학습이 끝난 후에는:

```text
MedGemma Base Model
        +
LoRA Adapter
        ↓
Fine-tuned MedGemma
        ↓
FastAPI
        ↓
POST /api/v1/ai/chat
```

구조로 사용할 수 있습니다.

**아래는 코드가 아니라 설명용 스니펫입니다 — 이 셀은 실행하지 마세요.** `base_model = ...`은
자리표시일 뿐 실제로 동작하는 코드가 아니고, 이건 이 학습 노트북이 아니라 나중에
`ai/llm`(현재 빈 패키지)에 서버 쪽 로딩 코드를 짤 때 참고할 형태입니다:

```python
from peft import PeftModel

base_model = ...  # 서버 쪽에서 medgemma-4b-it을 로드한 것

model = PeftModel.from_pretrained(
    base_model,
    "<HF계정>/medgemma-4b-lora-consultation"
)
```


---

# 37. 학습 데이터에 대한 주의사항

현재 사용하는:

```text
sean0042/KorMedMCQA
```

데이터셋은 실험 단계에서 사용할 수 있지만, **라이선스가 CC-BY-NC-2.0(비영리 조건)** 입니다.

실제 상용 서비스로 발전시키는 경우 데이터셋 라이선스와 모델 라이선스를 별도로 검토해야 합니다.

또한 실제 환자 개인정보가 포함된 데이터는 GitHub나 공개 Hugging Face Repository에 업로드하지
마세요.

---

# 38. 최종 전체 흐름

```text
Google Colab
    │
    ├── T4 GPU
    │
    ├── NumPy (Colab 기본값, 강제로 안 낮춤)
    │
    ├── KorMedMCQA
    │
    ├── MedGemma 4B
    │
    ├── 4bit Quantization
    │
    └── LoRA
            │
            ▼
       Fine-tuning
            │
            ▼
       Google Drive
       Checkpoint
            │
            ▼
       LoRA Adapter
            │
            ▼
      Hugging Face Hub
            │
            ▼
       FastAPI / Cloud Run
            │
            ▼
       실제 AI 서비스
```

# 핵심 체크리스트

* [ ] Colab GPU가 T4로 연결되어 있는가?
* [ ] numpy 버전을 억지로 낮추지 않고 Colab 기본값을 그대로 쓰고 있는가?
* [ ] 패키지 설치 후 런타임을 재시작했는가?
* [ ] `from datasets import load_dataset`가 정상 실행되는가?
* [ ] `pip check`에서 심각한 dependency conflict가 없는가?
* [ ] Hugging Face `HF_TOKEN`이 Colab Secrets에 등록되어 있는가?
* [ ] MedGemma 접근 권한이 있는가?
* [ ] GitHub clone은 현재 단계에서 생략했는가?
* [ ] Google Drive가 연결되어 있는가?
* [ ] T4에서 `fp16=False`, `bf16=False`로 설정했는가? (fp16 AMP는 Gemma 계열과 GradScaler 충돌 있음)
* [ ] 학습 전 Base Model 평가를 준비했는가?
* [ ] 학습 후 LoRA Adapter를 저장했는가?
* [ ] Hugging Face Repository를 private으로 설정했는가?
* [ ] 실제 서비스에서는 FastAPI가 Adapter를 불러오도록 구성했는가?


---

# 39. (선택) 재학습 없이 어댑터만 불러와서 빠르게 테스트

**세션이 끊겨서 다시 들어왔거나, 학습은 이미 끝나서 어댑터가 HF Hub에 올라가 있는 상태라면
이 섹션부터 실행하면 됩니다.** 데이터셋 로드/포맷팅(4~5번)이나 학습(9번) 전체를 다시 안 해도
됩니다 — 아래 5개 셀만 순서대로 실행하면 곧바로 테스트할 수 있습니다.

1. 패키지 설치(1번) → **런타임 다시 시작** → HF 로그인(2번)까지는 그대로 필요
2. 그다음 아래 셀들만 실행:


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "google/medgemma-4b-it"
# v1(gon-0130/medgemma-4b-lora-consultation)은 KorMedMCQA만 학습한 비교용으로 남겨뒀고,
# v2가 KorMedMCQA + GenMedGPT-5k-ko 혼합 학습한 최신 버전 — 옛날 v1을 테스트하고 싶으면
# 이 줄만 v1 이름으로 바꾸면 됨.
ADAPTER_REPO = "gon-0130/medgemma-4b-lora-consultation-v2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    token=HF_TOKEN,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
model.config.use_cache = True

print("어댑터 로드 완료")


## 39-1. 반복 증상("두두두...", "정정정...") 잡기 위한 디코딩 옵션

`use_cache`를 켜도 같은 증상이면, 캐시 문제가 아니라 **탐욕적(greedy) 디코딩이 학습으로 좁아진
확률분포에 갇혀서** 그럴 가능성이 큽니다(1 epoch·짧고 반복적인 학습 타깃 특성상 흔함).
`repetition_penalty`와 `no_repeat_ngram_size`로 같은 토큰 반복을 명시적으로 막고,
`do_sample=True`로 약간의 무작위성을 줘서 탈출 가능성을 높입니다.


In [ ]:
def ask(question):
    prompt = (
        "<start_of_turn>user\n"
        f"{question}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


print(ask("두통이 3일째 있어요. 어떤 진료과를 방문하는 것이 좋을까요?"))


**이래도 여전히 반복되면** 다음을 순서대로 시도한다.

1. `num_train_epochs`를 3으로 올려서 다시 학습 (단, loss가 비정상적으로 빨리 0에
   가까워지면 과적합이니 다시 낮춘다)
2. 그래도 안 되면, KorMedMCQA(객관식 시험 문제) 자체의 구조적 한계일 가능성이 크다 — 정답이
   아무리 완전한 문장으로 다듬어져도 결국 "N번, 짧은 명사구입니다" 패턴을 벗어나지 못해서,
   자유 서술형 상담 질문엔 근본적으로 안 맞을 수 있다. 이 경우엔 KorMedMCQA만 쓰지 말고
   `ChuGyouk/GenMedGPT-5k-ko`(환자 증상 설명 → 의사 진단/검사 추천, 자유 서술형, MIT
   라이선스) 같은 대화형 데이터셋을 섞어서(mixed dataset) 학습하는 걸 고려한다 — 상담
   챗봇이 실제로 내야 하는 출력 형태(문장 여러 개로 이어지는 자연스러운 답변)를 직접
   보여주는 데이터라, 정답 포맷을 아무리 다듬어도 KorMedMCQA 단독으로는 못 주는 신호다.
